In [2]:
import zipfile
with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

In [53]:
import pandas as pd
df = pd.read_csv('Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [54]:
# Preprocess the data
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

We have categorical variables like 'Gender' & 'Geography'

LabelEncoder on Geographical data will misinterpret the categories 

In [55]:
# Encode categorical variables
from sklearn.preprocessing import LabelEncoder
label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])
df.shape

(10000, 11)

In [56]:
# OHE for Geography Column 
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(df[['Geography']])
geo_encoder # Sparse stuff

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [57]:
# Array conversion
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [58]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [59]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [60]:
## Combine one hot encoded columns with the original data
data = pd.concat([df.drop('Geography', axis=1), geo_encoded_df], axis=1)

In [61]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [66]:
# LabelEncoder & OneHotEncoder were used to convert the Gender & Geography variables into numerical format
# Now saving it all in a pickle file
import pickle

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geopkl.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

# You are saving the transformation logic (fitted encoders)
# Used during deployement. Logic - [Male-0, Female-1]

In [63]:
## Divide the dataset into independent and dependent feature
X = data.drop('Exited', axis=1)
y = data['Exited']

## Split the data into train, test
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale these features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# X_train becomes a NumPy array, so you lose column names.
X_train = scaler.fit_transform(X_train)
X_train = pd.DataFrame(X_train, columns=X.columns)

X_test = scaler.transform(X_test)
X_test = pd.DataFrame(X_test, columns=X.columns)

In [64]:
X_train

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,0.356500,0.913248,-0.655786,0.345680,-1.218471,0.808436,0.649203,0.974817,1.367670,1.001501,-0.579467,-0.576388
1,-0.203898,0.913248,0.294938,-0.348369,0.696838,0.808436,0.649203,0.974817,1.661254,-0.998501,1.725723,-0.576388
2,-0.961472,0.913248,-1.416365,-0.695393,0.618629,-0.916688,0.649203,-1.025834,-0.252807,-0.998501,-0.579467,1.734942
3,-0.940717,-1.094993,-1.131148,1.386753,0.953212,-0.916688,0.649203,-1.025834,0.915393,1.001501,-0.579467,-0.576388
4,-1.397337,0.913248,1.625953,1.386753,1.057449,-0.916688,-1.540351,-1.025834,-1.059600,1.001501,-0.579467,-0.576388
...,...,...,...,...,...,...,...,...,...,...,...,...
7995,1.207474,0.913248,1.435808,1.039728,-0.102301,-0.916688,0.649203,0.974817,-0.539860,1.001501,-0.579467,-0.576388
7996,0.314989,-1.094993,1.816097,-1.389442,-1.218471,-0.916688,0.649203,0.974817,-1.733882,1.001501,-0.579467,-0.576388
7997,0.865009,-1.094993,-0.085351,-1.389442,-1.218471,2.533560,-1.540351,-1.025834,-0.142765,1.001501,-0.579467,-0.576388
7998,0.159323,0.913248,0.390011,1.039728,1.827259,-0.916688,0.649203,-1.025834,-0.050826,1.001501,-0.579467,-0.576388


In [65]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [16]:
with open('scaler.pkl', 'rb') as file:
    data = pickle.load(file)

In [17]:
data

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


### ANN Implemenatation

In [67]:
import tensorflow as tf
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime as dt

In [68]:
type((X_train.shape[1],))

tuple

In [69]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), ## HL1
    Dense(32, activation='relu'), ## Hidden Layer 2
    Dense(1, activation='sigmoid') ## Output Layer
])

C:\Users\dhrub\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [70]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [83]:
## In order to do forward and backward propagation 
## Comile the model
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.losses import BinaryCrossentropy

loss = BinaryCrossentropy
opt = Adam(learning_rate=0.01)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy", 
    metrics=['accuracy'])

In [85]:
# Set up the TensorBoard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

# TensorBoard: To visualize all logs that you have while training your model

# Creating a directory to store the model
log_dir = "log/better/" + dt.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [86]:
## Set up Early Stoppping

# after training model for 5-6 epochs out of 100 epochs
# .....model may train at its best and no furthur training amy be required
# ......monitored through loss value decrement

early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [87]:
## Train the model
history = model.fit(
    X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,
    callbacks=[tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8633 - loss: 0.3243 - val_accuracy: 0.8625 - val_loss: 0.3399
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8665 - loss: 0.3210 - val_accuracy: 0.8610 - val_loss: 0.3385
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8666 - loss: 0.3197 - val_accuracy: 0.8595 - val_loss: 0.3369
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8650 - loss: 0.3186 - val_accuracy: 0.8605 - val_loss: 0.3406
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8680 - loss: 0.3171 - val_accuracy: 0.8600 - val_loss: 0.3382
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8660 - loss: 0.3150 - val_accuracy: 0.8625 - val_loss: 0.3472
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8702 - loss: 0.3128 - val_accuracy: 0.8585 - val_loss: 0.3414
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8680 - loss: 0.3117 - val_accu

In [89]:
model.save('model2.h5')

In [90]:
# Load TensorBoard Extension
%reload_ext tensorboard

In [93]:
import os

print(os.listdir("logs"))

['fit']


You basically debugged a real-world ML tooling issue, which is honestly more valuable than just training a model.

🎯 What you just learned (this is important)

You now understand:

✅ How TensorBoard logging actually works
✅ Why callbacks are critical
✅ How log directory structure affects visualization
✅ How to debug when UI shows nothing despite logs existing
✅ That sometimes the issue is the tool, not your code

👉 This is exactly the kind of practical understanding that helps in internships + placements.

In [82]:
%tensorboard --logdir logs

Reusing TensorBoard on port 6007 (pid 26112), started 19:26:36 ago. (Use '!kill 26112' to kill it.)

In [102]:
%tensorboard --logdir log

In [ ]:
# Networking (TCP lifecycle)
# OS-level socket handling
# Web server behavior
# 👉 This is actual backend/system-level understanding
!kill 14504

kill: 14504: No such process


In [42]:
import tensorflow as tf
import datetime
import numpy as np
import shutil

# साफ कर दो logs
shutil.rmtree("logs", ignore_errors=True)

# data
X_train = np.random.rand(1000, 10)
y_train = np.random.randint(0, 2, 1000)

# model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 🔥 manual writer (THIS is the key)
log_dir = "logs/manual/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
writer = tf.summary.create_file_writer(log_dir)

# training loop
epochs = 5
for epoch in range(epochs):
    history = model.fit(X_train, y_train, epochs=1, verbose=0)

    loss = history.history['loss'][0]
    acc = history.history['accuracy'][0]

    # 🔥 write scalars manually
    with writer.as_default():
        tf.summary.scalar('loss', loss, step=epoch)
        tf.summary.scalar('accuracy', acc, step=epoch)

    writer.flush()

C:\Users\dhrub\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
tensorboard --logdir logs/